In [ ]:
# Prophet Forecasting for GPU Usage
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from prophet import Prophet

In [ ]:
# Load and prepare data
df_full = pd.read_csv(
    "../data/gpu_in_use.csv", parse_dates=["DTime"], index_col="DTime"
)

# Pre-shutdown data for training (excludes holiday shutdown Dec 19 - Jan 7)
df = df_full[df_full.index < "2025-12-19"]

# Smooth the data
df_daily = df.resample("D").mean()
df_smoothed = df_daily.ewm(span=10, adjust=False).mean()

# Full smoothed data for visualization
df_full_daily = df_full.resample("D").mean()
df_full_smoothed = df_full_daily.ewm(span=10, adjust=False).mean()

print(
    f"Training data: {df_smoothed.index.min()} to {df_smoothed.index.max()} ({len(df_smoothed)} days)"
)

In [ ]:
# Prepare data for Prophet (requires 'ds' and 'y' columns)
prophet_df = df_smoothed.reset_index().rename(columns={"DTime": "ds", "Usage": "y"})

# Fit Prophet model
model_prophet = Prophet(
    daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False
)
model_prophet.fit(prophet_df)
print("Prophet model fitted.")

In [ ]:
# Forecast 90 days (3 months) ahead
future = model_prophet.make_future_dataframe(periods=90, freq="D")
forecast_prophet = model_prophet.predict(future)

# Get only the future forecast part
forecast_future = forecast_prophet[
    forecast_prophet["ds"] > df_full_smoothed.index.max()
]

In [ ]:
# Plot: Historical data + Prophet forecast
fig = go.Figure()

# Full historical data (includes shutdown)
fig.add_trace(
    go.Scatter(
        x=df_full_smoothed.index,
        y=df_full_smoothed["Usage"],
        mode="lines",
        name="Actual (smoothed)",
        line=dict(color="blue"),
    )
)

# Prophet forecast
fig.add_trace(
    go.Scatter(
        x=forecast_future["ds"],
        y=forecast_future["yhat"],
        mode="lines",
        name="Prophet Forecast",
        line=dict(color="green"),
    )
)

# Confidence interval
fig.add_trace(
    go.Scatter(
        x=list(forecast_future["ds"]) + list(forecast_future["ds"][::-1]),
        y=list(forecast_future["yhat_upper"])
        + list(forecast_future["yhat_lower"][::-1]),
        fill="toself",
        fillcolor="rgba(0,255,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        name="95% CI",
    )
)

# Shutdown marker
fig.add_vrect(
    x0="2025-12-19",
    x1="2026-01-07",
    fillcolor="gray",
    opacity=0.2,
    annotation_text="Shutdown",
    annotation_position="top left",
)

fig.update_layout(
    title="Prophet: GPU Usage 3-Month Forecast",
    xaxis_title="Date",
    yaxis_title="Usage",
    yaxis=dict(range=[0, 4000], dtick=500),
    height=600,
)
fig.show()

In [ ]:
# Prophet built-in component visualization
fig_components = model_prophet.plot_components(forecast_prophet)

In [ ]:
# Evaluate Prophet on train/test split
split_idx = int(len(df_smoothed) * 0.8)
train = df_smoothed.iloc[:split_idx]
test = df_smoothed.iloc[split_idx:]

# Train Prophet on training data
prophet_train = train.reset_index().rename(columns={"DTime": "ds", "Usage": "y"})
prophet_eval = Prophet(
    daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False
)
prophet_eval.fit(prophet_train)

# Predict on test period
future_eval = prophet_eval.make_future_dataframe(periods=len(test), freq="D")
forecast_eval = prophet_eval.predict(future_eval)
prophet_pred = forecast_eval[forecast_eval["ds"].isin(test.index)]["yhat"].values

# Calculate metrics
mae = np.mean(np.abs(test["Usage"].values - prophet_pred))
rmse = np.sqrt(np.mean((test["Usage"].values - prophet_pred) ** 2))
mape = (
    np.mean(np.abs((test["Usage"].values - prophet_pred) / test["Usage"].values)) * 100
)

# Naive baseline
naive_pred = train["Usage"].iloc[-1]
naive_mae = np.mean(np.abs(test["Usage"] - naive_pred))

print(f"Prophet MAE: {mae:.2f}")
print(f"Prophet RMSE: {rmse:.2f}")
print(f"Prophet MAPE: {mape:.1f}%")
print(f"Naive MAE: {naive_mae:.2f}")
print(f"Improvement over naive: {(naive_mae - mae) / naive_mae * 100:.1f}%")